In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#note still have split_frac for safety
def prepare_data(ticker: str, start: str = '2016-01-01', seq_len: int = 30, split_frac: float = 0.8, test_frac: float = 0.2, val_frac: float = 0.15):
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna()
    test_split = int((1 - test_frac) * len(df)) #boundary: [train+val]|test
    val_split = int(test_split * (1 - val_frac)) #boundary: train|val
    split = int(split_frac * len(df))

    scaler = StandardScaler().fit(df['log_ret'].iloc[:split].values.reshape(-1, 1))
    scaled = scaler.transform(df['log_ret'].values.reshape(-1, 1)).flatten()

    def make_windows(s, seq):
        X = np.stack([s[i:i + seq] for i in range(len(s) - seq)])
        y = np.array([s[i + seq] for i in range(len(s) - seq)])
        return X, y

    Xtr_np, ytr_np = make_windows(scaled[:val_split], seq_len)
    Xval_np, yval_np = make_windows(scaled[val_split:test_split], seq_len)
    Xte_np, yte_np = make_windows(scaled[test_split:], seq_len)

    to_tensor = lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(-1).to(device)
    Xtr, ytr = to_tensor(Xtr_np), to_tensor(ytr_np)
    Xval, yval = to_tensor(Xval_np), to_tensor(yval_np)
    Xte, yte = to_tensor(Xte_np), to_tensor(yte_np)

    return {
        'Xtr': Xtr, 'ytr': ytr, 'Xval': Xval, 'yval': yval, 'Xte': Xte, 'yte': yte,
        'Xtr_np': Xtr_np, 'Xte_np': Xte_np,
        'scaler': scaler, 'df': df, 'seq_len': seq_len,
    }

In [2]:
class PredictionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super().__init__()
        self.hidden_dim, self.num_layers = hidden_dim, num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

In [3]:
# train model here
def train_model(Xtr, ytr, epochs=200, Xval=None, yval=None, hidden_dim=32, num_layers=2, lr=1e-3, print_every=25, patience=20):
    model = PredictionModel(1, hidden_dim, num_layers, 1).to(device)
    criterion = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        pred = model(Xtr)
        loss = criterion(pred, ytr)
        opt.zero_grad()
        loss.backward()
        opt.step()

        val_loss = None
        if Xval is not None:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(Xval), yval).item()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
 
        if epoch % print_every == 0:
            msg = f"  epoch {epoch:4d}  train_loss {loss.item():.6f}"
            if val_loss is not None:
                msg += f"  val_loss {val_loss:.6f}  (best {best_val_loss:.6f}, no_improve {epochs_no_improve})"
            print(msg)
 
        if Xval is not None and epochs_no_improve >= patience:
            print(f"  Early stopping at epoch {epoch} (no val improvement for {patience} epochs)")
            break
 
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"  Restored weights from best epoch (val_loss {best_val_loss:.6f})")
 
    return model

def train_linear_baseline(Xtr_np: np.ndarray, ytr_np: np.ndarray) -> LinearRegression:
    lr = LinearRegression()
    lr.fit(Xtr_np, ytr_np)
    return lr

In [4]:
def pct_vs_baseline(model_val: float, baseline_val: float, lower_is_better: bool):
    if baseline_val == 0:
        return None, "n/a (baseline is 0 by definition)"
    if lower_is_better:
        pct = (baseline_val - model_val) / abs(baseline_val) * 100
    else:
        pct = (model_val - baseline_val) / abs(baseline_val) * 100
    return pct, f"{pct:+.1f}%"  

In [5]:
METRIC_DIRECTION = {'rmse': True, 'mae': True, 'r2': False, 'directional_accuracy': False}

#All functions to evaluate model are placed in this code block
def evaluate(model, Xte, yte, scaler, linear_model=None, Xte_np=None) -> dict:
    model.eval()
    with torch.no_grad():
        pred_scaled = model(Xte).cpu().numpy()
    true_scaled = yte.cpu().numpy()
 
    pred = scaler.inverse_transform(pred_scaled).flatten()
    true = scaler.inverse_transform(true_scaled).flatten()
    naive = np.zeros_like(true)
 
    linear_pred = None
    if linear_model is not None and Xte_np is not None:
        linear_pred_scaled = linear_model.predict(Xte_np).reshape(-1, 1)
        linear_pred = scaler.inverse_transform(linear_pred_scaled).flatten()
 
    results = {'_pred': pred, '_true': true, '_linear_pred': linear_pred}
 
    def score(pred_arr, true_arr):
        return {
            'rmse': root_mean_squared_error(true_arr, pred_arr),
            'mae': mean_absolute_error(true_arr, pred_arr),
            'r2': r2_score(true_arr, pred_arr),
            'directional_accuracy': (np.sign(pred_arr) == np.sign(true_arr)).mean(),
        }
 
    model_scores = score(pred, true)
    naive_scores = score(naive, true)
    up_rate = (np.sign(true) > 0).mean()
    naive_scores['directional_accuracy'] = max(up_rate, 1 - up_rate)
    naive_scores['r2'] = 0.0
 
    linear_scores = score(linear_pred, true) if linear_pred is not None else None
 
    for m in ['rmse', 'mae', 'r2', 'directional_accuracy']:
        lower_better = METRIC_DIRECTION[m]
        pct_naive, label_naive = pct_vs_baseline(model_scores[m], naive_scores[m], lower_better)
        entry = {
            'model': model_scores[m], 'baseline_naive': naive_scores[m],
            'pct_vs_naive': pct_naive, 'pct_vs_naive_label': label_naive,
            'beats_naive': (model_scores[m] < naive_scores[m]) if lower_better else (model_scores[m] > naive_scores[m]),
        }
        if linear_scores is not None:
            pct_lin, label_lin = pct_vs_baseline(model_scores[m], linear_scores[m], lower_better)
            entry.update({
                'baseline_linear': linear_scores[m], 'pct_vs_linear': pct_lin, 'pct_vs_linear_label': label_lin,
                'beats_linear': (model_scores[m] < linear_scores[m]) if lower_better else (model_scores[m] > linear_scores[m]),
            })
        results[m] = entry
 
    results['prediction_variance_ratio'] = pred.var() / true.var() if true.var() > 0 else float('nan')
    if linear_pred is not None:
        results['linear_prediction_variance_ratio'] = linear_pred.var() / true.var() if true.var() > 0 else float('nan')
 
    return results

In [6]:
def print_report(results: dict):
    has_linear = results['rmse'].get('baseline_linear') is not None
    header = f"{'Metric':<22}{'Model':>10}{'vs Naive':>12}{'Beats?':>9}"
    if has_linear:
        header += f"{'vs Linear':>12}{'Beats?':>9}"
    print(f"\n{'='*len(header)}")
    print(header)
    print('-' * len(header))
    for name in ['rmse', 'mae', 'r2', 'directional_accuracy']:
        r = results[name]
        row = f"{name:<22}{r['model']:>10.4f}{r['pct_vs_naive_label']:>12}{('YES' if r['beats_naive'] else 'NO'):>9}"
        if has_linear:
            row += f"{r['pct_vs_linear_label']:>12}{('YES' if r['beats_linear'] else 'NO'):>9}"
        print(row)
    print('=' * len(header))
 
    pvr = results['prediction_variance_ratio']
    print(f"\nLSTM prediction variance / actual variance: {pvr:.3f}")
    if has_linear:
        print(f"Linear prediction variance / actual variance: {results['linear_prediction_variance_ratio']:.3f}")
    if pvr < 0.1:
        print(
            "The LSTM is likely just predicting values close to the mean "
            "rather than capturing real movement. Treat any RMSE/MAE win "
            "alongside this number, not in isolation."
        )
 

In [7]:
def mcnemar_directional_test(pred: np.ndarray, baseline_pred: np.ndarray, true: np.ndarray) -> dict:
    """
    PRIMARY test. Paired comparison on the SAME test days: does the LSTM
    get days right that the baseline gets wrong, more often than the
    reverse? Only disagreement days carry information -- days where both
    are right or both are wrong are excluded from the test statistic.
    """
    model_correct = np.sign(pred) == np.sign(true)
    baseline_correct = np.sign(baseline_pred) == np.sign(true)
 
    b = int((model_correct & ~baseline_correct).sum())   # LSTM right, baseline wrong
    c = int((~model_correct & baseline_correct).sum())   # baseline right, LSTM wrong
 
    table = [[0, b], [c, 0]]
    result = mcnemar(table, exact=(b + c < 25))
 
    return {
        'lstm_only_correct_days': b,
        'baseline_only_correct_days': c,
        'p_value': result.pvalue,
        'significant_at_0.05': result.pvalue < 0.05,
    }
 
 
def independent_subsample(pred: np.ndarray, true: np.ndarray, seq_len: int):
    """
    Overlapping test windows share up to (seq_len - 1) input days, so
    consecutive predictions are NOT independent trials -- a requirement the
    binomial test assumes. This keeps only predictions spaced >= seq_len
    apart, so kept windows share zero input days with each other.
    Costly (throws away most of the test set) but statistically honest.
    """
    idx = np.arange(0, len(pred), seq_len)
    return pred[idx], true[idx]
 
 
def binomial_secondary_test(pred: np.ndarray, true: np.ndarray, seq_len: int, p0: float = 0.5) -> dict:
    """
    SECONDARY test. On the independent subsample: is directional accuracy
    distinguishable from pure chance (p0=0.5)? Answers a more basic
    question than McNemar ("any edge at all?" vs. "edge over THIS baseline?").
    """
    pred_sub, true_sub = independent_subsample(pred, true, seq_len)
    n = len(pred_sub)
    if n < 5:
        return {'n_independent_trials': n, 'p_value': None,
                 'note': 'Too few independent trials to test reliably -- need a longer test period.'}
 
    correct = int((np.sign(pred_sub) == np.sign(true_sub)).sum())
    result = binomtest(correct, n, p=p0, alternative='greater')
 
    return {
        'n_independent_trials': n,
        'correct': correct,
        'accuracy': correct / n,
        'p0': p0,
        'p_value': result.pvalue,
        'significant_at_0.05': result.pvalue < 0.05,
    }

In [8]:
def print_significance_report(mcnemar_result: dict, binomial_result: dict):
    print(f"\n{'='*70}")
    print("SIGNIFICANCE TESTS")
    print('=' * 70)
 
    print("\nPRIMARY -- McNemar's test (LSTM vs. linear baseline, paired days):")
    print(f"  Days only LSTM got right:     {mcnemar_result['lstm_only_correct_days']}")
    print(f"  Days only baseline got right: {mcnemar_result['baseline_only_correct_days']}")
    print(f"  p-value: {mcnemar_result['p_value']:.4f}  "
          f"({'SIGNIFICANT' if mcnemar_result['significant_at_0.05'] else 'not significant'} at 0.05)")
 
    print("\nSECONDARY -- Binomial test (LSTM vs. pure chance, independent subsample):")
    if binomial_result.get('p_value') is None:
        print(f"  {binomial_result['note']}")
    else:
        print(f"  Independent trials: {binomial_result['n_independent_trials']}  "
              f"(accuracy: {binomial_result['accuracy']:.1%} vs. p0={binomial_result['p0']:.0%})")
        print(f"  p-value: {binomial_result['p_value']:.4f}  "
              f"({'SIGNIFICANT' if binomial_result['significant_at_0.05'] else 'not significant'} at 0.05)")
 
    print(f"\n{'='*70}")
    mcn_sig = mcnemar_result['significant_at_0.05']
    bin_sig = binomial_result.get('significant_at_0.05', False)
    if mcn_sig and bin_sig:
        verdict = "Strongest case: LSTM beats the baseline AND beats chance. Worth pursuing further."
    elif bin_sig and not mcn_sig:
        verdict = "LSTM beats chance, but isn't clearly better than the linear baseline -- the simpler model may capture the same edge."
    elif mcn_sig and not bin_sig:
        verdict = "Unusual combination -- double check whether the baseline itself is performing worse than chance."
    else:
        verdict = "No significant evidence of real predictive skill yet. Report honestly rather than tuning until it passes."
    print(f"Verdict: {verdict}")

In [9]:
if __name__ == "__main__":
    TICKER = "NTES"
    SEQ_LEN = 30
 
    print(f"Preparing data for {TICKER}")
    data = prepare_data(TICKER, seq_len=SEQ_LEN)
    print(f"Train windows: {len(data['Xtr'])}, Val windows: {len(data['Xval'])}, Test windows: {len(data['Xte'])}")
 
    print("Training LSTM (with early stopping on validation loss)")
    model = train_model(data['Xtr'], data['ytr'], Xval=data['Xval'], yval=data['yval'], epochs=500, patience=20)
 
    print("Training linear-regression baseline (same inputs)")
    linear_model = train_linear_baseline(data['Xtr_np'], data['ytr'].cpu().numpy().flatten())
 
    print("\nEvaluating model")
    results = evaluate(model, data['Xte'], data['yte'], data['scaler'],
                        linear_model=linear_model, Xte_np=data['Xte_np'])
    print_report(results)
 
    pred, true, linear_pred = results['_pred'], results['_true'], results['_linear_pred']
 
    mcnemar_result = mcnemar_directional_test(pred, linear_pred, true)
    binomial_result = binomial_secondary_test(pred, true, SEQ_LEN, p0=0.5)
    print_significance_report(mcnemar_result, binomial_result)
 
    deciles = pd.qcut(pred, 10, labels=False, duplicates='drop')
    up_rate_by_decile = pd.Series((true > 0).astype(float)).groupby(deciles).mean()
    print("\nUp-rate by prediction decile (0=lowest predicted, N=highest):")
    print(up_rate_by_decile)
 

Preparing data for NTES
Train windows: 1800, Val windows: 294, Test windows: 509
Training LSTM (with early stopping on validation loss)
  epoch    0  train_loss 1.047222  val_loss 0.801031  (best 0.801031, no_improve 0)
  epoch   25  train_loss 1.026484  val_loss 0.781655  (best 0.781655, no_improve 0)
  epoch   50  train_loss 1.024251  val_loss 0.778203  (best 0.778203, no_improve 0)
  epoch   75  train_loss 1.023437  val_loss 0.776511  (best 0.776481, no_improve 6)
  Early stopping at epoch 89 (no val improvement for 20 epochs)
  Restored weights from best epoch (val_loss 0.776481)
Training linear-regression baseline (same inputs)

Evaluating model

Metric                     Model    vs Naive   Beats?   vs Linear   Beats?
--------------------------------------------------------------------------
rmse                      0.0229       +0.1%      YES       +0.4%      YES
mae                       0.0162       +0.1%      YES       +0.9%      YES
r2                        0.0012n/a (bas